### __Git repository:__
https://github.com/KeremOzemre/Computational-Social-Science.git

#### __Main areas of contributions:__

##### __Part 1:__ s244794 20%, s245290 40%, s244322 40%
##### __Part 2:__ s244794 50%, s245290 25%, s244322 25%


In [ ]:
import pickle
import networkx as nx
import pandas as pd
import ast
from collections import Counter
import numpy as np

## __Part 1: Mixing Patterns and Assortativity__

### __Exercise 1: Mixing Patterns and Assortativity__

### __Part 1: Assortativity Coefficient__

##### 1. Calculate the Assortativity Coefficient for the network based on the country of each node. Implement the calculation using the formula provided during the lecture, also available in this paper (equation 2, here for directed networks). Do not use the NetworkX implementation.

In [ ]:
with open("G_social_compute.pkl", "rb") as f:
    G_social_compute = pickle.load(f)

for node in list(G_social_compute.nodes())[:5]:
    print(G_social_compute.nodes[node])

In [ ]:
from collections import defaultdict

mixing = defaultdict(int)

for u, v in SG.edges():

    cu = SG.nodes[u]['country']
    cv = SG.nodes[v]['country']

    mixing[(cu, cv)] += 1
    mixing[(cv, cu)] += 1
    
m = SG.number_of_edges()

for key in mixing:
    mixing[key] /= (2*m)
    
    
a = defaultdict(float)

for (i,j), val in mixing.items():
    a[i] += val

In [ ]:
sum_eii = sum(val for (i,j), val in mixing.items() if i == j)

sum_aibi = sum(a[i]*a[i] for i in a)

r = (sum_eii - sum_aibi) / (1 - sum_aibi)

print("Country assortativity:", r)

### __Part 2: Configuration model__ In the following, we are going to assess the significance of the assortativity by comparing the network's assortativity coefficient against that of random networks generated through the configuration model.

##### Implement the configuration model using the double edge swap algorithm to generate random networks. Ensure each node retains its original degree but with altered connections. Create a function that does that by following these steps:

##### __2.__ 
- a. Create an exact copy of your original network.
- b. Select two edges, and , ensuring u != y and v != x.
- c. Flip the direction of 
 to 
 50% of the time. This ensure that your final results is not biased, in case your edges were sorted (they usually are).
- d. Ensure that new edges 
 and 
 do not already exist in the network.
- e. Remove edges and 
 and add edges and 
- f. Repeat steps b to e until you have performed  swaps, where E is the total number of edges.


In [ ]:
import random

def double_edge_swap_algo(G, n_swaps=None, max_tries_per_swap=20, seed=None):
   
    if seed is not None:
        random.seed(seed)

    G_rand = G.copy()
    E = G_rand.number_of_edges()

    if n_swaps is None:
        n_swaps = 10 * E

    max_tries = max_tries_per_swap * n_swaps

    def canon(a, b):
        return (a, b) if a < b else (b, a)

    edges = [canon(u, v) for u, v in G_rand.edges()]
    edge_set = set(edges)
    edge_to_idx = {e: i for i, e in enumerate(edges)}

    successful_swaps = 0
    tries = 0

    while successful_swaps < n_swaps and tries < max_tries:
        tries += 1

        i1, i2 = random.sample(range(len(edges)), 2)

        u, v = edges[i1]
        x, y = edges[i2]

        if len({u, v, x, y}) < 4:
            continue

        # flip first edge 50% of the time
        if random.random() < 0.5:
            u, v = v, u

        # proposed new edges
        e1_new = canon(u, y)
        e2_new = canon(x, v)

        # avoid self-loops
        if u == y or x == v:
            continue

        # avoid duplicates / existing edges
        if e1_new in edge_set or e2_new in edge_set:
            continue
        
        e1_old = canon(u, v)
        e2_old = canon(x, y)
        
        if e1_old == e2_old:
            continue

        G_rand.remove_edge(*e1_old)
        G_rand.remove_edge(*e2_old)
        G_rand.add_edge(*e1_new)
        G_rand.add_edge(*e2_new)

        edge_set.remove(e1_old)
        edge_set.remove(e2_old)
        edge_set.add(e1_new)
        edge_set.add(e2_new)

        edges[i1] = e1_new
        edges[i2] = e2_new

        del edge_to_idx[e1_old]
        del edge_to_idx[e2_old]
        edge_to_idx[e1_new] = i1
        edge_to_idx[e2_new] = i2

        successful_swaps += 1

        if successful_swaps % 50000 == 0:
            print(f"{successful_swaps} successful swaps completed...")

    print(f"Finished: {successful_swaps} successful swaps out of requested {n_swaps}")
    print(f"Total attempts: {tries}")

    if successful_swaps < n_swaps:
        print("Warning: stopped before reaching requested swaps. You can increase max_tries_per_swap.")

    return G_rand

In [ ]:
G_rand = double_edge_swap_algo(
    G_social_compute,
    n_swaps=10 * G_social_compute.number_of_edges(),
    max_tries_per_swap=20,
    seed=42
)

##### __3. Double check that your algorithm works well, by showing that the degree of nodes in the original network and the new 'randomized' version of the network are the same.__

In [ ]:
original_degrees = dict(G_social_compute.degree())
randomized_degrees = dict(G_rand.degree())

print("Degrees preserved:", original_degrees == randomized_degrees)
print("Original nodes:", G_social_compute.number_of_nodes(), " Randomized nodes:", G_rand.number_of_nodes())
print("Original edges:", G_social_compute.number_of_edges(), " Randomized edges:", G_rand.number_of_edges())

### __Part 3: Analyzing Assortativity in Random Networks__

##### __4. Generate and analyze at least 100 random networks using the configuration model. For each, calculate the assortativity with respect to the country and plot the distribution of these values.__

In [ ]:
from collections import defaultdict

def calculate_country_assortativity(G, country_attr='country'):
    mixing = defaultdict(int)

    for u, v in G.edges():
        cu = G.nodes[u][country_attr]
        cv = G.nodes[v][country_attr]

        mixing[(cu, cv)] += 1
        mixing[(cv, cu)] += 1

    m = G.number_of_edges()

    for key in mixing:
        mixing[key] /= (2 * m)

    a = defaultdict(float)
    for (i, j), val in mixing.items():
        a[i] += val

    sum_eii = sum(val for (i, j), val in mixing.items() if i == j)
    sum_aibi = sum(a[i] * a[i] for i in a)

    r = (sum_eii - sum_aibi) / (1 - sum_aibi)
    return r

##### Original network country assortativity

In [ ]:
observed_country_r = calculate_country_assortativity(G_social_compute, country_attr='country')
print("Observed country assortativity:", observed_country_r)

##### Random Network Generation

In [ ]:
import time


random_country_assortativities = []

n_random_networks = 100
n_swaps = 10 * G_social_compute.number_of_edges()

start_time = time.time()

for i in range(n_random_networks):
    print(f"Random network {i+1}/{n_random_networks}")

    SG_rand = double_edge_swap_algo(
        SG,
        n_swaps=n_swaps,
        seed=42 + i
    )

    r_rand = calculate_country_assortativity(SG_rand, country_attr='country')
    random_country_assortativities.append(r_rand)

end_time = time.time()

print("Done.")
print(f"Total runtime: {end_time - start_time:.2f} seconds")

##### Plotting Social Computation Network's and Random Networks' Country Assortativity

In [ ]:
#plotter så  de 100 rnadom netwroks assotritvity ift. vores på 0.26

import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 5))
plt.hist(random_country_assortativities, bins=100, edgecolor='black')
plt.axvline(observed_country_r, linestyle='--', linewidth=2, label=f'Observed r = {observed_country_r:.3f}')
plt.xlabel("Country assortativity")
plt.ylabel("Frequency")
plt.title("Distribution of country assortativity in random networks")
plt.legend()
plt.show()

##### Compare the results with the assortativity of your original network to determine if connections within the same country are significantly higher than chance.

In [ ]:
random_mean = np.mean(random_country_assortativities)
random_std = np.std(random_country_assortativities)

print("Mean random assortativity:", random_mean)
print("Std random assortativity:", random_std)

In [ ]:
p_value = np.mean(np.array(random_country_assortativities) >= observed_country_r)
print("Empirical p-value:", p_value)

### __Part 4: Assortativity by Degree__

In [ ]:
def calculate_degree_assortativity(G):

    ku_list = []
    kv_list = []

    for u, v in G.edges():

        ku = G.degree(u)
        kv = G.degree(v)

        ku_list.append(ku)
        kv_list.append(kv)

    ku = np.array(ku_list)
    kv = np.array(kv_list)

    mean_ku = np.mean(ku)
    mean_kv = np.mean(kv)

    mean_ku_kv = np.mean(ku * kv)

    mean_ku2 = np.mean(ku**2)
    mean_kv2 = np.mean(kv**2)

    numerator = mean_ku_kv - (mean_ku * mean_kv)

    denominator = np.sqrt(mean_ku2 - mean_ku**2) * np.sqrt(mean_kv2 - mean_kv**2)

    r = numerator / denominator

    return r

##### __5. Calculate degree assortativity for your network using the formula discussed in the lecture.__

In [ ]:
observed_degree_r = calculate_degree_assortativity(G_social_compute)

print("Observed degree assortativity:", observed_degree_r)

##### __6. Compare your network's degree assortativity against that of 100 random networks generated via the configuration model. Analyze whether your network shows a tendency for high-degree scientists to connect with other high-degree scientists and vice versa.__

In [ ]:
random_degree_assortativities = []

n_random_networks = 100
n_swaps = 10 * G_social_compute.number_of_edges()

for i in range(n_random_networks):

    print(f"Random network {i+1}/{n_random_networks}")

    SG_rand = double_edge_swap_algo(
        G_social_compute,
        n_swaps=n_swaps,
        seed=100 + i
    )

    r_rand = calculate_degree_assortativity(SG_rand)

    random_degree_assortativities.append(r_rand)

##### Plotting by Degree Assortavity - Random and Social Computation Networks

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(random_degree_assortativities, bins=20, edgecolor='black')

plt.axvline(
    observed_degree_r,
    color='red',
    linestyle='--',
    linewidth=2,
    label=f'Observed r = {observed_degree_r:.3f}'
)

plt.xlabel("Degree assortativity")
plt.ylabel("Frequency")
plt.title("Distribution of degree assortativity in random networks")

plt.legend()
plt.show()

In [ ]:
print("Observed degree assortativity:", observed_degree_r)
print("Mean random assortativity:", np.mean(random_degree_assortativities))
print("Std random assortativity:", np.std(random_degree_assortativities))

### __Part 5: Reflection questions__

##### __7. Assortativity by degree. Were the results of the degree assortativity in line with your expectations? Why or why not?__

##### __8. Edge flipping. In the process of implementing the configuration model, you were instructed to flip the edges (e.g., changing from (u,v) to (v,u)) 50% of the time. Why do you think this step is included?__

##### __9. Distribution of assortativity in random networks. Describe the distribution of degree assortativity values you observed for the random networks. Was the distribution pattern expected? Discuss how the nature of random network generation (specifically, the configuration model and edge flipping) might influence this distribution and whether it aligns with theoretical expectations.__

## __Part 2: TF - IDF__

### __Exercise 1: TF-IDF and the Computational Social Science communities. The goal for this exercise is to find the words characterizing each of the communities of Computational Social Scientists. What you need for this exercise:__

- #####  The assignment of each author to their network community, and the degree of each author (Week 6, Exercise 4). This can be stored in a dataframe or in two dictionaries, as you prefer.


In [ ]:
with open("G_social_compute.pkl", "rb") as f:
    G_social_compute = pickle.load(f)

In [ ]:
louvain_social_compute = nx.community.louvain_communities(G_social_compute,seed = 100)

##### The degree of each author, and their community id are just assigned as new features to the social computation network

In [ ]:
for community_id, community_nodes in enumerate(louvain_social_compute):
    for node in community_nodes:
        G_social_compute.nodes[node]['community'] = community_id

for node, deg in G_social_compute.degree():
    G_social_compute.nodes[node]['degree'] = deg

##### Check if assignment is correct

In [ ]:
G_social_compute.nodes['https://openalex.org/A5013693751']

- #####  The tokenized abstract dataframe (Week 7, Exercise 2)

In [ ]:
tokenized_abstract = pd.read_csv('/Users/keremozemre/Library/CloudStorage/OneDrive-DanmarksTekniskeUniversitet/social compute/Computational-Social-Science/tokenized_abstract.csv')

##### __1. First, check out the wikipedia page for TF-IDF. Explain in your own words the point of TF-IDF.__

- ##### __What does TF stand for?__

- ##### __What does IDF stand for?__

##### __2. Now, we want to find out which words are important for each community, so we're going to create several large documents, one for each community. Each document includes all the tokens of abstracts written by members of a given community.__

In [ ]:
import ast
import pandas as pd

def abstracts_for_community_id(community_id):
    G = G_social_compute
    
    # Step 1: get authors in the community
    authors_in_community = [
        node for node in G.nodes if G.nodes[node]["community"] == community_id
    ]
    print(f"Number of authors in community {community_id}: {len(authors_in_community)}")
    
    # Step 2: filter papers that contain at least one author from the community
    papers = papers_dataset[
        papers_dataset["author_ids"].apply(lambda authors: any(a in authors_in_community for a in authors))
    ]
    print(f"Number of papers in community {community_id}: {len(papers)}")
    
    # Step 3: select tokenized abstracts for these papers
    tokens_series = tokenized_abstract.loc[
        tokenized_abstract["id"].isin(papers["id"]) &
        tokenized_abstract["tokens"].notna() &
        (tokenized_abstract["tokens"].str.len() > 0),
        "tokens"
    ]
    
    # Step 4: efficiently convert string-lists to real lists and flatten
    
    token_list = tokens_series.apply(ast.literal_eval).explode().tolist()
    
    return token_list

community_abstracts = {}

for id in range(229):
    community_abstracts[id] = abstracts_for_community_id(id)

##### __3. Now, we're ready to calculate the TF for each word. Use the method of your choice to find the top 5 terms within the top 5 communities (by number of authors).__

In [ ]:
social_compute_louvain_communities_size = {}
for i,community in enumerate(louvain_social_compute):
    social_compute_louvain_communities_size[i] = len(community)
social_compute_louvain_communities_size

In [ ]:
asc = {k: v for k, v in sorted(social_compute_louvain_communities_size.items(), key=lambda item: item[1])}
id_list = list(asc.keys())[-5:]

print(id_list)

In [ ]:
from collections import Counter
def TF_calculate(id_list):
    tf_id = {}
    for i in id_list:
        tokens = community_abstracts[i]
        denom = len(tokens)
        # Count occurrences
        term_counts = Counter(tokens)

        # Get the terms in a fixed order
        terms = list(term_counts.keys())

        counts_array = np.array([term_counts[t] for t in terms])
      
        tf_array = np.log(1 + counts_array)
       
        tf_dictionary = {}
        for term, tf in zip(terms, tf_array):
            tf_dictionary[term] = tf
        tf_id[i] = tf_dictionary
    return tf_id

TF_top_5 = TF_calculate(id_list)

In [ ]:
for id in id_list:
    top_terms = dict(
        sorted(TF_top_5[id].items(), key=lambda item: item[1], reverse=True)[:5]
    )
    print(f"These are the top 5 terms for Community {id} {top_terms}")

- ##### __Describe similarities and differences between the communities.__ 
- ##### __Why aren't the TFs not necessarily a good description of the communities?__ 

- ##### __Next, we calculate IDF for every word.__ 

In [ ]:
from collections import defaultdict

N = 229
doc_freq = defaultdict(int)

for cid in range(229):
    unique_words = set(community_abstracts[cid])
    for word in unique_words:
        doc_freq[word] += 1

idf_dict_all = {
    word: np.log(N / (1 + doc_freq[word]))+1
    for word in doc_freq
}

- ##### __What base logarithm did you use? Is that important?__ 

### __Exercise 2: The Wordcloud.__ It's time to visualize our results!

- ##### __Install the WordCloud module.__

In [ ]:
from wordcloud import WordCloud

- ##### __Now, create word-cloud for each community. Feel free to make it as fancy or non-fancy as you like.__
- ##### __Make sure that, together with the word cloud, you print the names of the top three authors in each community (see my plot above for inspiration).__

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i,id in enumerate(top_9_id_list):
    ax = axes[i]

    items = TFIDF[id].items()
    authors = top_9_id_dict[id]["authors"]

    terms_dict = {
        word: float(score)
        for word, score in items   
        if score > 0
    }

   # Create a nicer word cloud
    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='plasma',          
        contour_width=1,
        contour_color='black',
        max_words=500
    ).generate_from_frequencies(terms_dict)
    
    # Plot
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis('off')
    
    # Format authors nicely 
    authors_str = ", ".join(authors[:3])
    
    # Stylish title
    ax.set_title(
        f"Community {id}",
        fontsize=14,
        fontweight='bold',
        pad=10
    )
    
    # Subtitle (authors)
    ax.text(
        0.5, -0.1,
        f"Top authors: {authors_str}",
        ha='center',
        va='top',
        transform=ax.transAxes,
        fontsize=10,
        style='italic'
    )

# Remove empty subplots if <9 communities
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

- ##### __Comment on your results. What can you conclude on the different sub-communities in Computational Social Science?__

- ##### __Look up online the top author in each community. In light of your search, do your results make sense?__

### __Exercise 3: Computational Social Science__
- ##### __In light of your data-driven analysis, has your understanding of the field changed? How?__